# Entrenamiento local del modelo de Telco Churn

_Mismo pipeline que `docker-training/train.py`, pero corriendo en tu laptop con `uv` — sin Docker._

**Módulo 4 — MLOps y Cloud** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

---

Este notebook hace lo mismo que el contenedor que va a correr la VM en el paso siguiente del módulo, pero local:

1. **Lee** `raw/telco_churn.csv` desde Azure Blob Storage (subido en el notebook 02).
2. **Entrena** un `LogisticRegression` con preprocesamiento (`StandardScaler` + `OneHotEncoder`) en un `Pipeline`.
3. **Reporta** accuracy + ROC-AUC sobre un split test.
4. **Sube** el `.pkl` a `models/telco_churn_logreg.pkl` en el mismo container.

La idea es que el notebook y `train.py` sean **dos vistas del mismo código**: aquí lo iteras paso a paso para entender qué hace; el `train.py` es la versión empaquetada para CI + cloud.

## 1. Credenciales

El notebook lee `AZURE_STORAGE_CONNECTION_STRING` y `AZURE_STORAGE_CONTAINER` del `.env` de la raíz del repo. Hay dos formas de tenerlas ahí:

**Opción A — Terraform (recomendada, una sola fuente de verdad).** Terraform crea el storage account; expón la connection string con:

```bash
cd modulo-4-mlops-y-cloud/terraform
task deploy            # primera vez: backend:setup → init → apply → creds:export
# o si ya aplicaste:
task creds:export      # solo escribe el .env
```

Eso deja en `.env` (de la raíz):

```bash
AZURE_STORAGE_ACCOUNT=dsrpm4data<hex>
AZURE_STORAGE_CONTAINER=dsrp-modulo4
AZURE_STORAGE_CONNECTION_STRING="DefaultEndpointsProtocol=https;..."
```

**Opción B — manual con Azure CLI** (si todavía no quieres pasar por Terraform):

```bash
RG=dsrp-modulo4-rg
ACCOUNT=dsrpstorage$RANDOM    # debe ser único globalmente
az group create -n $RG -l eastus
az storage account create -n $ACCOUNT -g $RG -l eastus --sku Standard_LRS --kind StorageV2
az storage container create --name dsrp-modulo4 --account-name $ACCOUNT
az storage account show-connection-string -n $ACCOUNT -g $RG -o tsv
# pega la salida en .env
```

> El `.env` está en `.gitignore`. **Nunca lo commitees**.

In [ ]:
import io
import os
import platform
import socket
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from azure.storage.blob import BlobServiceClient, ContentSettings
from dotenv import load_dotenv
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

load_dotenv()  # carga .env de la raíz

CONN_STR = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
CONTAINER = os.environ.get("AZURE_STORAGE_CONTAINER", "dsrp-modulo4")
INPUT_BLOB = "raw/telco_churn.csv"
OUTPUT_BLOB = "models/telco_churn_logreg.pkl"
TARGET = "Churn"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Mismos helpers de logging que train.py (en el contenedor van a journalctl;
# acá los vemos inline en la celda).
def _log(msg=""): print(f"[nb] {msg}")
def _section(title):
    bar = "─" * (len(title) + 4)
    print(f"\n[nb] {bar}\n[nb]  {title}\n[nb] {bar}")

_section("RUNTIME")
_log(f"run_id     = {RUN_ID}")
_log(f"host       = {socket.gethostname()}")
_log(f"python     = {platform.python_version()}")
_log(f"sklearn    = {sklearn.__version__}")
_log(f"container  = azure://{CONTAINER}")
_log(f"input      = {INPUT_BLOB}")
_log(f"output     = {OUTPUT_BLOB}")

## 2. Cliente de Blob y helpers (los del notebook 02)

Reusamos exactamente los mismos helpers que escribimos en `02_azure_blob_storage_sdk.ipynb`. Aquí los repetimos para que el notebook sea autocontenido — en un repo de producción los moverías a `src/dsrp/storage.py` e importarías.

In [ ]:
service = BlobServiceClient.from_connection_string(CONN_STR)
container_client = service.get_container_client(CONTAINER)

# Idempotente: crea el container si no existe (Terraform ya lo crea, pero por si acaso).
if not container_client.exists():
    container_client.create_container()
    print(f"Container '{CONTAINER}' creado.")
else:
    print(f"Container '{CONTAINER}' ya existe.")


def download_blob_to_df(blob_name: str) -> pd.DataFrame:
    """Descarga un blob CSV directo a un DataFrame en memoria."""
    bc = service.get_blob_client(container=CONTAINER, blob=blob_name)
    data = bc.download_blob().readall()
    return pd.read_csv(io.BytesIO(data))


def upload_file_as_blob(local_path: Path, blob_name: str, content_type: str = "application/octet-stream") -> str:
    """Sube un archivo local como blob. Devuelve la URL."""
    bc = service.get_blob_client(container=CONTAINER, blob=blob_name)
    with local_path.open("rb") as f:
        bc.upload_blob(f, overwrite=True, content_settings=ContentSettings(content_type=content_type))
    return bc.url


print("Helpers listos:", download_blob_to_df.__name__, upload_file_as_blob.__name__)

## 3. Leer el CSV desde Blob Storage

Si esto falla con `BlobNotFound`, regresa al notebook 02 y corre la celda de upload — el contenedor del trainer (y este notebook) asumen que `raw/telco_churn.csv` ya existe.

In [ ]:
_section("DESCARGA")
t0 = time.perf_counter()
df = download_blob_to_df(INPUT_BLOB)
dl_s = time.perf_counter() - t0
_log(f"shape         : {df.shape}")
_log(f"download_s    : {dl_s:.2f}")
_log(f"df memoria_mb : {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f}")
_log(f"nulos totales : {df.isna().sum().sum():,}")

_section("BALANCE DE CLASES")
for cls, count in df[TARGET].value_counts().items():
    _log(f"  {cls:<5} {count:>5,}  ({count / len(df):.1%})")

df.head(3)

## 4. Preprocesamiento

El dataset de Telco tiene un detalle conocido: `TotalCharges` viene como string con espacios en algunas filas (~11 de 7043). Lo coercionamos a número y descartamos esas filas — es la misma lógica que en `train.py`.

In [ ]:
_section("PREPROCESAMIENTO")

def preprocess(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    df = df.copy()
    n_before = len(df)
    df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
    n_invalid = df["TotalCharges"].isna().sum()
    df = df.dropna(subset=["TotalCharges"])
    _log(f"TotalCharges  : {n_invalid} filas no-numéricas descartadas ({n_invalid / n_before:.2%})")
    df = df.drop(columns=["customerID"])
    _log(f"dropped       : customerID")
    y = (df[TARGET] == "Yes").astype(int)
    X = df.drop(columns=[TARGET])
    return X, y


X, y = preprocess(df)
_log(f"X shape       : {X.shape}")
_log(f"y churn rate  : {y.mean():.3%}")
X.dtypes.value_counts()

## 5. Pipeline de sklearn

`ColumnTransformer` separa numéricas vs categóricas → `StandardScaler` para las numéricas, `OneHotEncoder(handle_unknown="ignore")` para las categóricas → `LogisticRegression` al final. Empaquetado en un `Pipeline` para que el `.pkl` resultante sea autosuficiente (no necesita preprocesar a mano al hacer `predict`).

In [ ]:
_section("PIPELINE")

def build_pipeline(X: pd.DataFrame) -> tuple[Pipeline, dict]:
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(include=["object"]).columns.tolist()
    _log(f"numéricas ({len(numeric)})    : {numeric}")
    _log(f"categóricas ({len(categorical)}) : {categorical}")
    pre = ColumnTransformer(
        [
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ]
    )
    pipe = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, n_jobs=-1))])
    return pipe, {"numeric": numeric, "categorical": categorical}


X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipe, pipe_meta = build_pipeline(X_tr)

_section("ENTRENAMIENTO")
_log(f"X_train       : {X_tr.shape}")
_log(f"X_test        : {X_te.shape}")
t0 = time.perf_counter()
pipe.fit(X_tr, y_tr)
fit_s = time.perf_counter() - t0
n_features_out = pipe.named_steps["pre"].transform(X_tr.head(1)).shape[1]
_log(f"features OHE  : {n_features_out}")
_log(f"fit duration_s: {fit_s:.2f}")

## 6. Métricas

Lo mismo que el trainer imprime en `journalctl` cuando corre en la VM. Si los números aquí y en la VM no coinciden a 3 decimales, es señal de que algo cambió en el dataset (alguien subió un CSV distinto al blob).

In [ ]:
_section("MÉTRICAS")

# Train
proba_tr = pipe.predict_proba(X_tr)[:, 1]
preds_tr = (proba_tr >= 0.5).astype(int)
train_acc = accuracy_score(y_tr, preds_tr)
train_auc = roc_auc_score(y_tr, proba_tr)

# Test
proba_te = pipe.predict_proba(X_te)[:, 1]
preds_te = (proba_te >= 0.5).astype(int)
test_acc = accuracy_score(y_te, preds_te)
test_auc = roc_auc_score(y_te, proba_te)
test_prec = precision_score(y_te, preds_te)
test_rec = recall_score(y_te, preds_te)
test_f1 = f1_score(y_te, preds_te)
cm = confusion_matrix(y_te, preds_te)

_log(f"train accuracy : {train_acc:.4f}")
_log(f"train roc_auc  : {train_auc:.4f}")
_log(f"test  accuracy : {test_acc:.4f}")
_log(f"test  roc_auc  : {test_auc:.4f}")
_log(f"test  precision: {test_prec:.4f}")
_log(f"test  recall   : {test_rec:.4f}")
_log(f"test  f1       : {test_f1:.4f}")
_log(f"overfit gap    : {train_acc - test_acc:+.4f}")
_log("confusion matrix (rows=actual, cols=pred):")
_log(f"             pred_no_churn  pred_churn")
_log(f"  no_churn  {cm[0, 0]:>13,d}  {cm[0, 1]:>10,d}")
_log(f"  churn     {cm[1, 0]:>13,d}  {cm[1, 1]:>10,d}")

metrics = {
    "train_accuracy": float(train_acc),
    "train_roc_auc": float(train_auc),
    "test_accuracy": float(test_acc),
    "test_roc_auc": float(test_auc),
    "test_precision": float(test_prec),
    "test_recall": float(test_rec),
    "test_f1": float(test_f1),
    "confusion_matrix": cm.tolist(),
    "n_features_out": int(n_features_out),
    "train_seconds": fit_s,
}

## 7. Serializar y subir a Blob Storage

El `train.py` empaqueta `{"model": pipe, "metrics": {...}}` para que cualquier consumidor pueda saber con qué métricas se entrenó este artefacto sin re-correr nada. Mismo formato aquí.

In [ ]:
_section("SERIALIZACIÓN + UPLOAD")
out = Path("/tmp/telco_churn_logreg.pkl")
bundle = {
    "model": pipe,
    "metrics": metrics,
    "pipeline_metadata": pipe_meta,
    "training_run": {
        "run_id": RUN_ID,
        "input_blob": INPUT_BLOB,
        "output_blob": OUTPUT_BLOB,
        "dataset_rows": int(len(df)),
        "train_rows": int(len(X_tr)),
        "test_rows": int(len(X_te)),
        "sklearn_version": sklearn.__version__,
        "python_version": platform.python_version(),
    },
}
joblib.dump(bundle, out)
_log(f"pickle path   : {out}")
_log(f"pickle bytes  : {out.stat().st_size:,}")

t0 = time.perf_counter()
url = upload_file_as_blob(out, OUTPUT_BLOB)
up_s = time.perf_counter() - t0
_log(f"upload_s      : {up_s:.2f}")
_log(f"url           : {url}")

## 8. Verificar que quedó arriba

In [ ]:
for blob in container_client.list_blobs(name_starts_with="models/"):
    print(f"  - {blob.name:<40} {blob.size:>10,} bytes   {blob.last_modified:%Y-%m-%d %H:%M}")

## 9. ¿Y ahora qué?

Acabas de hacer **manualmente** lo que el contenedor Docker hace **automatizado**. El siguiente paso es "empaquetar" este flujo:

1. **Misma lógica, en `train.py`** → ya está en `docker-training/train.py`. Compáralo: es prácticamente las mismas funciones de este notebook.
2. **Empaquetado en Docker** → `docker-training/Dockerfile` lo encierra con sus dependencias. Pruébalo localmente:
   ```bash
   cd modulo-4-mlops-y-cloud/docker-training
   task build-run     # build + docker run leyendo el mismo .env
   ```
3. **CI publica la imagen** → push a `main` cualquier cambio bajo `docker-training/` y la GitHub Action `build-modulo4-image.yml` publica `ghcr.io/<user>/dsrp-modulo4-trainer:latest`.
4. **Terraform levanta una VM que ejecuta esa imagen** → `cd ../terraform && task deploy`. Si quieres que corra periódicamente, define `trainer_schedule` en `dsrp-values.tfvars` (ej. `"hourly"` o `"*-*-* 03:00:00"`) — la VM instalará un systemd timer.

El notebook es para entender; el resto es para que el trabajo no dependa de que tú lo corras a mano.